In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hammadansari7/e-commerce-orders-and-customer/E-Commerce Orders.csv.xlsx


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'iframe'
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
# We will use Seaborn as a fallback because it is Kaggle's most stable viz tool
sns.set_theme(style="darkgrid")



# Setting a professional dark theme for all interactive plots
pio.templates.default = "plotly_dark"

# Load the dataset
df = pd.read_excel('/kaggle/input/datasets/hammadansari7/e-commerce-orders-and-customer/E-Commerce Orders.csv.xlsx')

# Initial Pre-processing
df['Date'] = pd.to_datetime(df['Date'])
df['MonthYear'] = df['Date'].dt.to_period('M').astype(str)
df['DayOfWeek'] = df['Date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print("Setup Complete. Dataset loaded with", len(df), "records.")

Setup Complete. Dataset loaded with 1200 records.


In [3]:
fig1 = px.scatter(df.sort_values('Date'), x="UnitPrice", y="TotalPrice", animation_frame="MonthYear", 
                 size="Quantity", color="Product", hover_name="Product",
                 log_x=True, size_max=45, range_x=[10,1000], range_y=[0,4000],
                 title="Product Revenue Dynamics Over Time (Animated)")
fig1.show()

In [4]:
fig2 = px.sunburst(df, path=['ReferralSource', 'PaymentMethod', 'OrderStatus'], 
                  values='TotalPrice', color='TotalPrice', color_continuous_scale='Viridis',
                  title="Revenue Streams: Marketing Source -> Payment -> Order Status")
fig2.show()

In [5]:
status_counts = df['OrderStatus'].value_counts().reset_index()
fig3 = px.funnel(status_counts, x='count', y='OrderStatus', title="Order Lifecycle Funnel")
fig3.show()

In [6]:
fig4 = px.violin(df, y="UnitPrice", x="Product", color="Product", box=True, points="all",
                title="Unit Price Density & Outliers per Product Category")
fig4.show()

In [7]:
corr = df.select_dtypes(include=[np.number]).corr()
fig5 = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale='RdBu_r',
                title="Interactive Feature Correlation Matrix")
fig5.show()

In [8]:
day_source = df.groupby(['DayOfWeek', 'ReferralSource'])['TotalPrice'].mean().reset_index()
fig6 = px.line_polar(day_source, r='TotalPrice', theta='DayOfWeek', color='ReferralSource', 
                    line_close=True, title="Avg Revenue per Day by Referral Channel",
                    category_orders={"DayOfWeek": day_order})
fig6.show()

In [9]:
fig7 = px.scatter_3d(df, x='Quantity', y='ItemsInCart', z='TotalPrice',
                    color='Product', size='UnitPrice', opacity=0.7,
                    title="3D Behavioral Cube: Quantity vs Cart Items vs Total Revenue")
fig7.show()

In [10]:
total_rev = df['TotalPrice'].sum()
fig8 = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = total_rev,
    title = {'text': "Total Lifetime Revenue ($)"},
    gauge = {'axis': {'range': [0, 1500000]}, 'bar': {'color': "#00CC96"}}))
fig8.show()

In [11]:
s_map = {val: i for i, val in enumerate(df['ReferralSource'].unique())}
p_map = {val: i + len(s_map) for i, val in enumerate(df['Product'].unique())}
fig9 = go.Figure(data=[go.Sankey(
    node = dict(label = list(s_map.keys()) + list(p_map.keys())),
    link = dict(source = df['ReferralSource'].map(s_map), 
                target = df['Product'].map(p_map), 
                value = df['TotalPrice']))])
fig9.update_layout(title_text="Purchase Flow: Referral Source to Product Category")
fig9.show()

In [12]:
fig10 = px.density_heatmap(df, x="DayOfWeek", y="Product", z="TotalPrice", 
                          histfunc="sum", title="Sales Intensity Heatmap (Product vs Day)",
                          category_orders={"DayOfWeek": day_order})
fig10.show()

In [13]:
# Convert categories to numbers
le = LabelEncoder()
df_ml = df.copy()
cat_cols = ['Product', 'PaymentMethod', 'ReferralSource', 'OrderStatus']

for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

# Select features and target
X = df_ml[['Product', 'Quantity', 'UnitPrice', 'PaymentMethod', 'ReferralSource', 'ItemsInCart']]
y = df_ml['TotalPrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data prepared for Machine Learning.")

Data prepared for Machine Learning.


In [14]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print(f"Model Performance:")
print(f"R-Squared: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:.2f}")

Model Performance:
R-Squared: 0.9998
Mean Absolute Error: $7.71


In [15]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
fig_ml = px.bar(importances, orientation='h', 
               title="Machine Learning Insights: Drivers of Revenue",
               labels={'value':'Importance Score', 'index':'Feature'})
fig_ml.show()